# Five-qubit quantum error correction

The `[[5,1,3]]` perfect code stores one logical qubit in five physical qubits and corrects any one-qubit Pauli error. This notebook uses `src/five_qubit_code.py`, which builds circuits in the same Qiskit style as CNCSim's circuit examples, reuses the shared Pauli/symplectic helpers, and checks stabilizer syndromes with the existing `ChpSimulator`.

For arbitrary logical amplitudes, the notebook uses Qiskit's exact `Statevector` simulator. The CNC `run_qcm` QASM path currently has a restricted initial-state and feed-forward interface, so it is not used for this workflow.

In [ ]:
import numpy as np
from qiskit.quantum_info import Statevector

from src.five_qubit_code import (
    LOGICAL_X, LOGICAL_Z, STABILIZERS,
    encode_state, error_correction_circuit,
    measure_syndrome_with_chp, simulate_recovery,
    syndrome_for_error,
)

## Code definition and qubit convention

The generators are `XZZXI`, `IXZZX`, `XIXZZ`, and `ZXIXZ`. The logical operators are `X_L = XXXXX` and `Z_L = ZZZZZ`; they commute with every stabilizer and anticommute with each other. In CNCSim's Pauli-string convention, the leftmost character is data qubit 0.

Each syndrome bit is ordered by its stabilizer: `(S1, S2, S3, S4)`. A bit is 1 exactly when the measured stabilizer eigenvalue is -1, meaning the error anticommutes with that generator.

In [ ]:
print('Stabilizers:', STABILIZERS)
print('Logical X:', LOGICAL_X)
print('Logical Z:', LOGICAL_Z)
print('Example encoded circuit:')
encode_state((1 / np.sqrt(2), 1 / np.sqrt(2))).draw('text')

## Encode an arbitrary logical state

The encoder takes the logical input on data qubit 0 and four `|0>` ancillas on data qubits 1 through 4. Here the input has a nontrivial relative phase, so the example is not limited to a computational-basis stabilizer state. The returned circuit prepares that qubit and applies the synthesized Clifford encoder.

In [ ]:
logical_state = (np.sqrt(0.37), np.exp(0.7j) * np.sqrt(0.63))
encoded_circuit = encode_state(logical_state)
encoded_state = Statevector.from_instruction(encoded_circuit)
print('Encoded state has', encoded_state.num_qubits, 'physical qubits')
print('Circuit operations:', encoded_circuit.count_ops())

## Inject errors, measure syndromes, and recover

The first table shows the unique syndrome assigned to each of the 15 non-identity single-qubit Pauli errors. `measure_syndrome_with_chp` runs the extraction circuit on encoded `|0_L>` with CNCSim's CHP simulator. The next cell uses exact statevector evolution for an arbitrary logical state, applies the syndrome-decoded recovery, then checks both logical-state fidelity and all four stabilizer expectations.

In [ ]:
syndrome_table = {
    f'{pauli}{qubit}': syndrome_for_error(qubit, pauli)
    for qubit in range(5) for pauli in 'XYZ'
}
syndrome_table

In [ ]:
examples = ((0, 'X'), (2, 'Y'), (4, 'Z'))
for qubit, pauli in examples:
    chp_syndrome = measure_syndrome_with_chp(qubit, pauli)
    result = simulate_recovery(logical_state, qubit, pauli)
    print(f'{pauli} error on data qubit {qubit}')
    print('  syndrome (S1..S4):', result.syndrome)
    print('  CHP syndrome:', chp_syndrome)
    print('  recovery:', result.correction)
    print(f'  logical-state fidelity: {result.fidelity:.12f}')
    print('  stabilizer expectations:', result.stabilizer_expectations)

## Compose an executable recovery circuit

`error_correction_circuit()` returns a nine-qubit Clifford circuit: data qubits 0–4, syndrome ancillas 5–8, and four classical syndrome bits. It measures each stabilizer and uses a Qiskit classical condition to apply the matching X, Y, or Z correction. It resets its syndrome ancillas after recovery so additional checks can reuse them.

In [ ]:
correction = error_correction_circuit()
print(correction.num_qubits, 'qubits;', correction.num_clbits, 'classical syndrome bits')
correction.draw('text')

## Validation notes

The tests in `tests/test_five_qubit_code.py` check the stabilizer and logical-operator algebra, validate the encoder, and run all 15 single-qubit errors through three checks: unique symplectic syndromes, measured syndromes from `ChpSimulator`, and conditional recovery in Qiskit Aer followed by a second syndrome extraction. They also compare the recovered arbitrary logical state against its ideal encoded state using exact fidelity.